# 00 - Setup & Data Exploration

CSE-547 Deep Learning Final Project  
Object Recognition using RGB and InfraRed images (Autonomous Driving)

This notebook:
1. Verifies environment (PyTorch + GPU)
2. Generates manifests for IR and RGB data
3. Validates the video-level train/val split
4. Explores class distributions and sample images

## 1. Environment Check

In [ ]:
import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import os

print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Generate Manifests

In [ ]:
from data_manifest import generate_all

ir_df, rgb_df, split_df, paired_df = generate_all(val_fraction=0.2, seed=42)

## 3. Validate Split (No Leakage)

In [ ]:
for name, df in [("IR", ir_df), ("RGB", rgb_df)]:
    train_vids = set(df[df["split"] == "train"]["video_id"])
    val_vids = set(df[df["split"] == "val"]["video_id"])
    overlap = train_vids & val_vids
    total = len(df)
    train_n = (df["split"] == "train").sum()
    val_n = (df["split"] == "val").sum()
    print(f"{name}: {total:,} total | {train_n:,} train ({len(train_vids)} vids) | "
          f"{val_n:,} val ({len(val_vids)} vids) | Video overlap: {len(overlap)}")
    assert len(overlap) == 0, f"{name} has video leakage!"

## 4. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (name, df) in zip(axes, [("IR", ir_df), ("RGB", rgb_df)]):
    counts = df.groupby(["split", "class_name"]).size().unstack(fill_value=0)
    counts.T.plot(kind="bar", ax=ax, width=0.8)
    ax.set_title(f"{name} Class Distribution by Split")
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")
    ax.legend(title="Split")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("figures/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Sample Images

In [ ]:
from data_manifest import CLASSES

# Show sample IR patches (one per class)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, class_name in zip(axes.flat, sorted(CLASSES.keys())):
    samples = ir_df[ir_df["class_name"] == class_name].head(1)
    if len(samples) > 0:
        img = Image.open(samples.iloc[0]["patch_path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"IR: {class_name}", fontsize=12)
    ax.axis("off")

fig.suptitle("Sample IR Patches (one per class)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Verify 10x Parameter Growth Rule

In [ ]:
from models import verify_10x_rule
from torchinfo import summary

param_counts = verify_10x_rule()

In [ ]:
# Detailed summary of each architecture
from models import build_cnn

for arch in ["A", "B", "C", "D"]:
    model, count = build_cnn(arch)
    print(f"\n{'='*60}")
    print(f"Architecture {arch} ({count:,} params)")
    print(f"{'='*60}")
    summary(model, input_size=(1, 3, 64, 64), verbose=1)

## 7. Paired Scene Evaluation Set

In [ ]:
print(f"Paired validation scenes: {len(paired_df)}")
if len(paired_df) > 0:
    print(f"\nSample:")
    display(paired_df.head(10))
    print(f"\nTotal RGB patches in paired val scenes: {paired_df['rgb_patch_count'].sum():,}")
    print(f"Total IR patches in paired val scenes: {paired_df['ir_patch_count'].sum():,}")

## 8. Quick DataLoader Test

In [ ]:
from training import get_dataloaders

# Test IR dataloader
train_loader, val_loader, _, _ = get_dataloaders(
    "manifests/ir_manifest.csv", batch_size=64, num_workers=0
)

images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Label values: {labels[:10]}")
print(f"Image range: [{images.min():.3f}, {images.max():.3f}]")
print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

---
**Setup complete.** Manifests, splits, and dataloaders are ready.  
Proceed to `01_part1_cnn_IR.ipynb` for Part 1 experiments.